# MNIST Classifier with MLP - tracked using MLFlow

## Step 0 — Setup
Install dependencies (skip if already installed) and import libraries.

In [ ]:
import mlflow
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:5001


## Step 1 — The starter script (un-instrumented)
This is the "before" version — plain scikit-learn, no tracking at all. Run it once just to confirm it works.

In [3]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=True, parser="auto")
X = X.astype("float32")/255
y = y.astype("int64")

X, _, y, _ = train_test_split(X, y, train_size=10000, stratify=y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def train_and_evaluate(learning_rate_init=0.001, batch_size=128, hidden_layer_sizes=(128,), max_iter=30):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        batch_size=batch_size,
        max_iter=max_iter,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=max_iter,
        random_state=42,
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    return model, acc, f1

# Sanity check — no MLflow involved yet
_, acc, f1 = train_and_evaluate()
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=0.9390  f1_macro=0.9385


/media/WindowsC/Users/vrish/Documents/IIT/AIDA/Sem 5/DA3408 - AI Ops Lab/aiops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


## Step 2 & 3 — Instrument it: manual logging
Wrap training in `with mlflow.start_run():` and log parameters, metrics, and a tag.

In [4]:
def train_and_log(learning_rate_init=0.001, batch_size=128, hidden_layer_sizes=(128,), max_iter=30, run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- parameters (at least 3) ---
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("random_state", 42)

        model, acc, f1 = train_and_evaluate(learning_rate_init, batch_size, hidden_layer_sizes, max_iter)

        for step, (loss, val_acc) in enumerate(zip(model.loss_curve_, model.validation_scores_)):
            mlflow.log_metric("train_loss", loss, step=step)
            mlflow.log_metric("val_accuracy", val_acc, step=step)

        # --- metrics (at least 2) ---
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(model, name="model", skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"])

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  acc={acc:.4f}  f1={f1:.4f}")
        return run_id

## Step 4 — Sweep: 6 runs varying lr and batch size
Re-run with different values and log each as its own MLflow run.

In [5]:
sweep_run_ids = []
for lr in [0.0003, 0.003, 0.03]:
    for bs in [32, 256]:
        rid = train_and_log(learning_rate_init=lr, batch_size=bs,
                            run_name=f"mlp-lr{lr}-bs{bs}")
        sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)

/media/WindowsC/Users/vrish/Documents/IIT/AIDA/Sem 5/DA3408 - AI Ops Lab/aiops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 5a55d6c35c834bec82cbb4965bff374d  |  acc=0.9345  f1=0.9339
🏃 View run mlp-lr0.0003-bs32 at: http://localhost:5001/#/experiments/2/runs/5a55d6c35c834bec82cbb4965bff374d
🧪 View experiment at: http://localhost:5001/#/experiments/2


/media/WindowsC/Users/vrish/Documents/IIT/AIDA/Sem 5/DA3408 - AI Ops Lab/aiops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 14460b2ab7484e1ab9d9a83d66becfd1  |  acc=0.9210  f1=0.9201
🏃 View run mlp-lr0.0003-bs256 at: http://localhost:5001/#/experiments/2/runs/14460b2ab7484e1ab9d9a83d66becfd1
🧪 View experiment at: http://localhost:5001/#/experiments/2


/media/WindowsC/Users/vrish/Documents/IIT/AIDA/Sem 5/DA3408 - AI Ops Lab/aiops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


...

Logged run e2a9fcc0718045fab2a533b5dc477719  |  acc=0.9395  f1=0.9391
🏃 View run mlp-lr0.003-bs256 at: http://localhost:5001/#/experiments/2/runs/e2a9fcc0718045fab2a533b5dc477719
🧪 View experiment at: http://localhost:5001/#/experiments/2


/media/WindowsC/Users/vrish/Documents/IIT/AIDA/Sem 5/DA3408 - AI Ops Lab/aiops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 94144334fb1d4a5ebbacfdb0bb9aaf16  |  acc=0.9125  f1=0.9118
🏃 View run mlp-lr0.03-bs32 at: http://localhost:5001/#/experiments/2/runs/94144334fb1d4a5ebbacfdb0bb9aaf16
🧪 View experiment at: http://localhost:5001/#/experiments/2


/media/WindowsC/Users/vrish/Documents/IIT/AIDA/Sem 5/DA3408 - AI Ops Lab/aiops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run df27bf1a46254a13880c33609c91a9d7  |  acc=0.9325  f1=0.9321
🏃 View run mlp-lr0.03-bs256 at: http://localhost:5001/#/experiments/2/runs/df27bf1a46254a13880c33609c91a9d7
🧪 View experiment at: http://localhost:5001/#/experiments/2
Sweep run IDs: ['5a55d6c35c834bec82cbb4965bff374d', '14460b2ab7484e1ab9d9a83d66becfd1', '42b6147b313f4224bd2d1f4b7230d4b0', 'e2a9fcc0718045fab2a533b5dc477719', '94144334fb1d4a5ebbacfdb0bb9aaf16', 'df27bf1a46254a13880c33609c91a9d7']


## Step 5 — Find the best run with `mlflow.search_runs()`
No need to open the UI to find the winner — query it directly.

In [6]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-classifier"],
    order_by=["metrics.accuracy DESC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.learning_rate_init", "params.batch_size", "metrics.accuracy", "metrics.f1_macro", "metrics.final_train_loss", "metrics.best_val_accuracy"
)]
print(runs_df[display_cols].to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}  (accuracy={best_run['metrics.accuracy']:.4f})")

                          run_id  metrics.accuracy  metrics.f1_macro params.batch_size params.learning_rate_init tags.mlflow.runName
42b6147b313f4224bd2d1f4b7230d4b0            0.9485          0.948123                32                     0.003    mlp-lr0.003-bs32
e2a9fcc0718045fab2a533b5dc477719            0.9395          0.939104               256                     0.003   mlp-lr0.003-bs256
5a55d6c35c834bec82cbb4965bff374d            0.9345          0.933947                32                    0.0003   mlp-lr0.0003-bs32
df27bf1a46254a13880c33609c91a9d7            0.9325          0.932067               256                      0.03    mlp-lr0.03-bs256
14460b2ab7484e1ab9d9a83d66becfd1            0.9210          0.920069               256                    0.0003  mlp-lr0.0003-bs256
94144334fb1d4a5ebbacfdb0bb9aaf16            0.9125          0.911758                32                      0.03     mlp-lr0.03-bs32

Best run: 42b6147b313f4224bd2d1f4b7230d4b0  (accuracy=0.9485)


## Run Comparison

In [7]:
client = mlflow.MlflowClient()
for _, r in runs_df.iterrows():
    tl = [m.value for m in client.get_metric_history(r.run_id, "train_loss")]
    va = [m.value for m in client.get_metric_history(r.run_id, "val_accuracy")]
    peak = va.index(max(va))
    print(f"{r['tags.mlflow.runName']:<20} loss {tl[0]:.3f}->{tl[-1]:.4f} | "
          f"val peak {max(va):.4f}@ep{peak} end {va[-1]:.4f} | drop {va[-1]-max(va):+.4f}")

mlp-lr0.003-bs32     loss 0.466->0.0029 | val peak 0.9525@ep21 end 0.9475 | drop -0.0050
mlp-lr0.003-bs256    loss 0.805->0.0047 | val peak 0.9400@ep18 end 0.9387 | drop -0.0012
mlp-lr0.0003-bs32    loss 1.018->0.0289 | val peak 0.9337@ep20 end 0.9337 | drop +0.0000
mlp-lr0.03-bs256     loss 0.869->0.1203 | val peak 0.9500@ep20 end 0.9213 | drop -0.0287
mlp-lr0.0003-bs256   loss 1.923->0.1592 | val peak 0.9263@ep26 end 0.9250 | drop -0.0012
mlp-lr0.03-bs32      loss 0.591->0.2215 | val peak 0.9200@ep23 end 0.9100 | drop -0.0100
